In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import scipy.stats as stats
from sklearn.impute import KNNImputer
import math
import re
from sklearn.metrics  import mean_squared_error, r2_score # for regression
from sklearn.linear_model import Lasso, LassoCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report # for classification
import xgboost as xgb


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 0. Get data

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
submission_ex = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

In [ ]:
# Save PassengerId before dropping for the submission
passengerid_test = test['PassengerId'].copy()

# 1. Data Cleaning

## 1.1. Inspect dataset

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
test.shape

In [ ]:
test.info()

In [ ]:
test.head()

In [ ]:
train.isnull().sum()

### 1.1.1. Check why the data is missing for the 'Age' variable

In [ ]:
msno.matrix(train)
msno.heatmap(train)

In [ ]:
# Compare the mean of the missing and non-missing groups, 0 = not missing, 1 = missing
train['Age_missing'] = train['Age'].isnull().astype(int)
train.groupby('Age_missing')['Survived'].mean()

In [ ]:
missing = train[train['Age'].isnull()]['Survived']
not_missing = train[train['Age'].notnull()]['Survived']

t_stats, p_value = stats.ttest_ind(missing, not_missing)
print(p_value)

## 1.2. Feature Engineering

In [ ]:
train['Name']

### 1.2.1. Mapping Titles

In [ ]:
def extract_titles(name):
    match = re.search(r' ([A-Za-z]+)\.', name) # Find titles from names
    title = match.group(1) if match else 'None'
    title_map = { 'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', 
                 'Rev': 'Rare', 'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare',
                'Lady': 'Rare', 'Sir': 'Rare', 'Col': 'Rare', 'Countess': 'Rare',
                'Jonkheer': 'Rare', 'Capt': 'Rare', 'Dona': 'Rare'}
    return title_map.get(title, title)

### 1.2.2. Process Pipeline -- Feature Engineering before dropping

In [ ]:
train.columns

In [ ]:
def process(df):
    df = df.copy()
    df['Title'] = df['Name'].apply(extract_titles)
    
    df['Family_Size'] = df['SibSp'] + df['Parch'] + 1  # add 1 = add themselves
    df['IsAlone'] = (df['Family_Size'] == 1).astype(int)
    
    df['Deck'] = df['Cabin'].fillna('U').str[0]
    df['Deck'] = df['Deck'].replace(['T'], 'Other')
    
    df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})
    
    df['FarePerPerson'] = df['Fare'] / df['Family_Size']
    
    df['Embarked'] = df['Embarked'].fillna('S')
    
    # Drop unused features
    df = df.drop(columns = ['Cabin', 'Ticket', 'PassengerId', 'Name'])

    # One-hot encode
    df = pd.get_dummies(df, columns = ['Title', 'Deck', 'Embarked'], 
                        drop_first = True, dtype = int)
    
    return df

train_processed = process(train)
test_processed = process(test)

print('train_processed: ', train_processed.shape)
print('test_processed: ', test_processed.shape)

In [ ]:
# Divide the train dataset into X (features) and y (predicting outcome)
X = train_processed.drop(columns = ['Survived'])
y = train_processed['Survived']

test_aligned = test_processed.reindex(columns = X.columns, fill_value = 0)

print('Match: ', list(X.columns) == list(test_aligned.columns))

### 1.2.4. Imputation for 'Age' feature

In [ ]:
# KNN Imputation
imputer = KNNImputer(n_neighbors = 20)

X = pd.DataFrame(imputer.fit_transform(X), columns = X.columns, index = X.index)
test_aligned = pd.DataFrame(imputer.transform(test_aligned), columns = test_aligned.columns, 
                           index = test_aligned.index)

print('train_null: ', X.isnull().sum().sum())
print('test_null: ', test_aligned.isnull().sum().sum())
print('test rows: ', len(test_aligned))

# 2. Exporatory Data Analsis (EDA)

## 2.0. Summary Statistics

In [ ]:
train.describe().round(3)

## 2.1. Check Distributions

In [ ]:
# Distribution of Sex
sns.countplot(x = 'Sex', data= train)

In [ ]:
# Distribution using histogram -- discret
train[['Age', 'Pclass', 'SibSp', 'Fare', 'Parch']].hist(figsize= (12,8), 
                                                        layout =(3,2), 
                                                        bins = 'fd')

## 2.2. Correlation 

### 2.2.1. Box plots

In [ ]:
# Box Plot (Age vs Survival: numeric vs binary)
sns.scatterplot(x = "Age", y = "Survived", data = train)

In [ ]:
# Scatter plot between Fare and Survived
sns.scatterplot(x = "Fare", y = "Survived", data = train)

### 2.2.2 Crosstab

In [ ]:
# Surviving rate with 'Sex' feature --- Raw numbers 
# Survived: 0 = No, 1 = Yes
pd.crosstab(train['Sex'], train['Survived'])

In [ ]:
# Surviving rate with 'Sex' feature --- Proportions
pd.crosstab(train['Sex'], train['Survived'], normalize = 'index')

In [ ]:
# Surviving rate with 'Age' feature --- Raw numbers 
# Survived: 0 = No, 1 = Yes
pd.crosstab(train['Survived'], train['Age'])

In [ ]:
# Surviving rate with 'Age' feature --- Proportion
pd.crosstab(train['Survived'], train['Age'], normalize = 'index')

### 2.2.3. Correlation

In [ ]:
# Correlation between 'Age' and 'Survived'
train['Age'].corr(train['Survived'])

In [ ]:
# Correlation between 'Fare' and 'Survived'
train['Fare'].corr(train['Survived'])

In [ ]:
train_processed.corr()

In [ ]:
sns.heatmap(train_processed.corr(), annot = False, fmt = '.2f')

# 3. Start with Lasso (L1 Logistic)

In [ ]:
# Split test and train data 
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = 0.2, 
                                                    random_state = 42, stratify = y)

In [ ]:
# Lasso processing pipeline
lasso_pipeline = Pipeline([('scaler', StandardScaler()), 
                          ('model', LogisticRegression(penalty= 'l1',
                                                      solver = 'liblinear', 
                                                      C = 1.0, max_iter = 1000))])
# Train the lasso model with train dataset
lasso_pipeline.fit(X_train, y_train)
print(f'Accuracy scores: {accuracy_score(y_test, lasso_pipeline.predict(X_test)):.4f}')

# Using 5-folds CV get accuacy scores of lasso model
lasso_cv = cross_val_score(lasso_pipeline, X, y, cv = 5, scoring = 'accuracy')
print(f'CV score: {lasso_cv.mean():.4f} ± {lasso_cv.std():.4f}')

# 4. Random Forest

## 4.1. Tuning: Grid Search using best parameters

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300, 400], 'max_depth': [None, 5, 10, 20, 30], 
    'min_samples_split': [2, 5, 10, 20], 'max_features': ['sqrt', 'log2']
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state = 42),
                          param_grid, cv = 5, scoring = 'accuracy', n_jobs =-1)

In [ ]:
rf_grid.fit(X_train, y_train)
print(f'Best Parameters: {rf_grid.best_params_}')

best_rf = rf_grid.best_estimator_
print(f'Accuracy: {accuracy_score(y_test, best_rf.predict(X_test)):.4f}')

rf_cv = cross_val_score(best_rf, X, y, cv = 5, scoring = 'accuracy')
print(f'CV score: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

### 4.1.1. Feature Importance

In [ ]:
importance_df = pd.DataFrame({'features': X.columns, 
                              'importance' : best_rf.feature_importances_
                             }).sort_values('importance', ascending = False)

print(importance_df)

In [ ]:
# Plot the covariates in descending order (the most important features are on the top)
plt.figure(figsize = (10,6))
plt.barh(importance_df['features'], 
         importance_df['importance'])
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title("Features in Descending Order")
plt.tight_layout()

# 5. XGboost

## 5.1. Train model and evaluate

In [ ]:
XGBoost = xgb.XGBClassifier(n_estimators = 400, 
                            subsample = 0.8,
                            learning_rate = 0.05,
                            max_depth = 4,
                            colsample_bytree = 0.8,
                            eval_metric = 'logloss',
                           random_state = 42)

XGBoost.fit(X_train, y_train)
y_pred_xgb = XGBoost.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')

CV_scores = cross_val_score(XGBoost, X, y, cv = 5, scoring = 'accuracy')
print(f'CV Accuracy: {CV_scores.mean():.4f} ± {CV_scores.std():.4f}')

print(classification_report(y_test, y_pred_xgb))

## 5.2. Tune & Optimizing: Early Stopping

In [ ]:
# Grid Search
grid_params = {'n_estimators': [100, 200, 300, 400, 500],
               'max_depth': [3,5,7,9],
               'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3, 0.4],
               'subsample': [0.5, 0.8, 1.0],
               'colsample_bytree': [0.5, 0.8, 1.0]}

In [ ]:
xgb_grid = GridSearchCV(xgb.XGBClassifier(), grid_params, 
                         cv = 5, scoring = 'accuracy', n_jobs = -1)

xgb_grid.fit(X_train, y_train)

In [ ]:
print('Best params:', xgb_grid.best_params_)
print('Best score:', xgb_grid.best_score_)

## 5.3. Features Importance

In [ ]:
xgb_importance_df = pd.DataFrame({'features': X.columns, 
                                  'importance': xgb_grid.best_estimator_.feature_importances_}).sort_values('importance',
                                                                                          ascending = False)
print(xgb_importance_df)

In [ ]:
xgb.plot_importance(xgb_grid.best_estimator_, max_num_features = 10)
plt.show()

# 6. Compare Models: Lasso, RF, XGBoost

In [ ]:
compare_models = {
    'Lasso' : lasso_pipeline,
    'Random Forest': best_rf,
    'XGBoost': xgb_grid.best_estimator_,
}

print(f"{'Model':<25} {'Model Acc':>10} {'Std':>8} {'Stability':>12}")
print('-'*60)

result = {}
for name, model in compare_models.items():
    scores = cross_val_score(model, X, y, cv = 5, scoring = 'accuracy')
    result[name] = scores.mean()
    stability = 'stable' if scores.std() < 0.02 else 'variable'
    print(f'{name:<25} {scores.mean():.4f} ± {scores.std():.4f} {stability:>12}')

best_model_name = max(result, key = result.get)
best_model = compare_models[best_model_name]
print(f'\nBest Model: {best_model_name}')

# 7. Predict Outcomes

In [ ]:
best_model.fit(X, y)
prediction = best_model.predict(test_aligned)
print(f'length of the prediction: {len(prediction)}')

In [ ]:
submission = pd.DataFrame({
    'PassengerId': passengerid_test,
    'Survived': prediction.astype(int)
})

In [ ]:
submission.to_csv('submission.csv', index = False)
print(submission.head())
print(len(submission))